### Accuracy Issue on Imbalanced Data

We see that about 88% of the data has "no" as the label, so scoring on accuracy may be misleading, and that is in fact reflected in the low precision/recall values in the classification report and confusion matrix in the term_deposit notebook. There are several fixes to this:

1. **Alternative scoring metrics.** Replace accuracy with a metric that is sensitive to the minority class — such as F1-score, ROC-AUC, or average precision (PR-AUC) — when comparing models during spot-checking and hyperparameter tuning. Accuracy rewards majority-class performance almost by default when classes are this imbalanced, so it should not be used as the sole model-selection criterion.

2. **Class weighting.** Many `sklearn` classifiers (e.g. `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier`, `SVC`) accept a `class_weight='balanced'` argument, which re-weights the loss function inversely proportional to class frequency, penalizing misclassification of the minority class more heavily during training. For algorithms without a built-in `class_weight` parameter (e.g. XGBoost), a similar effect can be achieved with `scale_pos_weight = (# negative) / (# positive)`.

3. **Decision threshold tuning.** Rather than using the default 0.5 probability threshold, the classification threshold can be chosen post hoc (e.g. via the precision-recall curve) to better balance precision and recall according to the real costs of false positives vs. false negatives in this business context.

4. **Resampling.** Alternatively (or in addition), the class distribution itself can be rebalanced before training — either by undersampling the majority class, oversampling the minority class, or synthetically generating new minority-class samples.

There is yet another complementary framework called `imbalanced-learn` (`imblearn`), purpose-built for exactly this type of problem — classification tasks with imbalanced classes. We use the `SMOTE` (Synthetic Minority Oversampling Technique) method from `imblearn` to synthetically generate new minority-class ("yes") samples by interpolating between existing minority-class points in feature space, rather than simply duplicating them.

Crucially, `SMOTE` is inserted as a step inside an `imblearn` pipeline, placed *after* preprocessing/feature-engineering and *immediately before* the classifier. This ensures oversampling is applied only to the training folds during cross-validation, never to the validation or test folds — avoiding data leakage that would otherwise produce artificially inflated performance estimates.

We use this SMOTE-augmented pipeline to spot-check the same 8 algorithms again, this time scoring on **F1** rather than accuracy, to see whether the "top three models" change. We then re-run hyperparameter tuning and final evaluation on this corrected pipeline, to see whether the final model choice — and, more importantly, its real-world performance on the minority class — changes as a result.

In [ ]:
!python -m pip install imbalanced-learn

In [ ]:
# If not already installed:
# !pip install imbalanced-learn

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (RobustScaler, QuantileTransformer, PolynomialFeatures,
                                    FunctionTransformer, LabelEncoder, OneHotEncoder, OrdinalEncoder)
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import numpy as np
import pandas as pd

bank_df = pd.read_csv('data/bank-full.csv', delimiter=';')

# ---- 1. Recreate the stratified split ----
strat_train_set, strat_test_set = train_test_split(
    bank_df, test_size=0.2, stratify=bank_df["y"], random_state=42)

X_train = strat_train_set.drop(columns='y', axis=1)
y_train = strat_train_set.y
X_test = strat_test_set.drop(columns='y', axis=1)
y_test = strat_test_set.y

# ---- 2. preprocessing functions from term deposit notebook, unchanged ----
def mark_val_missing(df):
    df.loc[df['poutcome']=='other', 'poutcome'] = np.nan
    df.loc[df['contact']=='unknown', 'contact'] = np.nan
    df.loc[df['pdays']== -1, 'pdays'] = 0
    return df

def impute(df):
    mode_impute = SimpleImputer(strategy='most_frequent')
    bank_df_to_impute = df.select_dtypes(exclude=[np.number])
    num_df = df.select_dtypes(include=[np.number])
    bank_df_inputed = mode_impute.fit_transform(bank_df_to_impute)
    bank_tr = pd.DataFrame(bank_df_inputed, columns=mode_impute.feature_names_in_, index=bank_df_to_impute.index)
    return pd.concat([bank_tr, num_df], axis=1)

def type_cast(df):
    num_df = df.select_dtypes(include=[np.number])
    cat_df = df.select_dtypes(exclude=[np.number]).astype(str)
    return pd.concat([cat_df, num_df], axis=1)

def encoder(df):
    ordinal_cols = ['month', 'poutcome']
    one_hot_cols = ['job','education','default','marital','housing','loan','contact']
    ordinal_df = df[ordinal_cols]
    one_hot_df = df[one_hot_cols]
    months = ['jan','feb','mar','apr','may','jun','jul','aug','sep','oct','nov','dec']
    outcomes = ['success','unknown','failure']
    ord_enc = OrdinalEncoder(categories=[months, outcomes])
    one_hot_enc = OneHotEncoder()
    ordinal_df_encoded = ord_enc.fit_transform(ordinal_df)
    one_hot_df_encoded = one_hot_enc.fit_transform(one_hot_df)
    merged_encoded = np.concatenate((one_hot_df_encoded.toarray(), ordinal_df_encoded), axis=1)
    enc_feature_names = one_hot_enc.get_feature_names_out().tolist() + ord_enc.get_feature_names_out().tolist()
    encoded_df = pd.DataFrame(merged_encoded, columns=enc_feature_names, index=df.index)
    df_num = df.select_dtypes(include=[np.number])
    return pd.concat([encoded_df, df_num], axis=1)

mark_val_missing_transformer = FunctionTransformer(mark_val_missing)
impute_transformer = FunctionTransformer(impute)
type_cast_transformer = FunctionTransformer(type_cast)
encoder_transformer = FunctionTransformer(encoder)

# ---- 3. Encode target (same as your cell 124) ----
preprocessing_target = LabelEncoder()
y_train_enc = preprocessing_target.fit_transform(y_train)
y_test_enc = preprocessing_target.transform(y_test)   # transform, not fit_transform — reuse the same mapping

# ---- 4. Column selection + scaling (computed after mark/impute/type_cast/encoder run) ----
# We need columns_to_trans based on the processed frame's int64 columns, same logic as your notebook:
X_train_probe = encoder(type_cast(impute(mark_val_missing(X_train.copy()))))
columns_to_trans = X_train_probe.select_dtypes(include=['int64']).columns

preprocessing = ColumnTransformer(
    transformers=[
        ('quantile_transform', QuantileTransformer(n_quantiles=100, output_distribution='normal'), columns_to_trans),
        ('scaling', RobustScaler(), columns_to_trans)
    ],
    remainder='passthrough'
)

# ---- 5. ONE full pipeline, exact steps, SMOTE inserted right before the classifier ----
def make_full_pipeline(clf):
    return ImbPipeline([
        ('mark_val_missing', mark_val_missing_transformer),
        ('impute', impute_transformer),
        ('type_cast', type_cast_transformer),
        ('encoder', encoder_transformer),
        ('preprocessing', preprocessing),
        ('poly_features_eng', PolynomialFeatures(degree=2)),
        ('dim_reduction', PCA(n_components=65)),
        ('smote', SMOTE(random_state=42)),
        ('classif', clf)
    ])

# ---- 6. Spot-check original 8 models, now with SMOTE + f1 scoring ----
classifiers = [
    ('XGB', xgb.XGBClassifier()),
    ('LR', LogisticRegression()),
    ('LDA', LinearDiscriminantAnalysis()),
    ('KNN', KNeighborsClassifier()),
    ('CART', DecisionTreeClassifier()),
    ('NB', GaussianNB()),
    ('SVM', SVC()),
    ('RFC', RandomForestClassifier())
]

cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)

for name, clf in classifiers:
    pipe = make_full_pipeline(clf)
    scores = cross_val_score(pipe, X_train, y_train_enc, cv=cv, scoring='f1', n_jobs=-1)
    print(f'{name}: Mean F1 = {scores.mean():.4f}, Std = {scores.std():.4f}')

# ---- 7. Hyperparameter tuning on Random Forest, same SMOTE pipeline ----
rf_pipeline = make_full_pipeline(RandomForestClassifier(random_state=42))

param_grid = {
    'classif__n_estimators': [100, 200, 300],
    'classif__max_depth': [None, 10, 20],
    'classif__min_samples_split': [2, 5, 10],
}

grid_search = GridSearchCV(rf_pipeline, param_grid=param_grid, scoring='f1', cv=5, n_jobs=-1)
grid_search.fit(X_train, y_train_enc)
print("Best params:", grid_search.best_params_)
best_model = grid_search.best_estimator_

# ---- 8. Final evaluation on the real, untouched, still-imbalanced test set ----
y_pred = best_model.predict(X_test)
y_hat = preprocessing_target.inverse_transform(y_pred)

print(classification_report(y_test, y_hat))
print(confusion_matrix(y_test, y_hat))

XGB: Mean F1 = 0.5625, Std = 0.0161
LR: Mean F1 = 0.3959, Std = 0.0141
LDA: Mean F1 = 0.4869, Std = 0.0116
KNN: Mean F1 = 0.4940, Std = 0.0163
CART: Mean F1 = 0.4470, Std = 0.0169
NB: Mean F1 = 0.2963, Std = 0.0176
SVM: Mean F1 = 0.3058, Std = 0.0144
RFC: Mean F1 = 0.5819, Std = 0.0152
Best params: {'classif__max_depth': 20, 'classif__min_samples_split': 5, 'classif__n_estimators': 200}
              precision    recall  f1-score   support

          no       0.96      0.90      0.93      7985
         yes       0.50      0.73      0.59      1058

    accuracy                           0.88      9043
   macro avg       0.73      0.81      0.76      9043
weighted avg       0.91      0.88      0.89      9043

[[7217  768]
 [ 290  768]]
